In [ ]:
!pip install pyspark

In [ ]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName('Widgets').getOrCreate()

In [ ]:
from google.colab import files, output

#enabling interactive widgets in google colab
output.enable_custom_widget_manager()

import pandas as pd
import ipywidgets as widgets
from IPython.display import display

#upload
uploaded = files.upload()

file_name = next(iter(uploaded))
df = pd.read_csv(file_name)

print("Dataset loaded successfully!")
print("Number of records:" , len(df))

Saving manufacturing_production_120k.csv to manufacturing_production_120k (1).csv
Dataset loaded successfully!
Number of records: 120000


In [ ]:
# Function to create dropdown options
def create_options(column, all_label):
    values = sorted(df[column].dropna().astype(str).unique())

    return [(all_label, "__ALL__")] + [
        (value, value) for value in values
    ]
    # 1. Plant dropdown
plant_dropdown = widgets.Dropdown(
    options=create_options("plant_id", "All Plants"),
    description="Plant:",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="260px")
)
    # 2. Product family dropdown
product_dropdown = widgets.Dropdown(
    options=create_options("product_family", "All Products"),
    description="Product Family:",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="300px")
)


# 3. Quality status dropdown
quality_dropdown = widgets.Dropdown(
    options=create_options("quality_status", "All Statuses"),
    description="Quality:",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="260px")
)


# 4. Slider for units produced
units = pd.to_numeric(df["units_produced"], errors="coerce").dropna()

units_slider = widgets.IntRangeSlider(
    value=(int(units.min()), int(units.max())),
    min=int(units.min()),
    max=int(units.max()),
    step=1,
    description="Units Produced:",
    continuous_update=False,
    style={"description_width": "initial"},
    layout=widgets.Layout(width="90%")
)
from IPython.display import display
from google.colab import output

# Enable interactive widgets in Google Colab
output.enable_custom_widget_manager()

# Display all four widgets
display(widgets.VBox([
    plant_dropdown,
    product_dropdown,
    quality_dropdown,
    units_slider
]))


def filter_data(plant, product, quality, unit_range):
    mask = pd.Series(True, index=df.index)

    if plant != "__ALL__":
        mask &= df["plant_id"].astype(str) == plant

    if product != "__ALL__":
        mask &= df["product_family"].astype(str) == product

    if quality != "__ALL__":
        mask &= df["quality_status"].astype(str) == quality

    units = pd.to_numeric(df["units_produced"], errors="coerce")
    mask &= units.between(unit_range[0], unit_range[1])

    filtered = df[mask]
    print("Number of filtered records:", len(filtered))
    display(filtered)

out = widgets.interactive_output(
    filter_data,
    {
        "plant": plant_dropdown,
        "product": product_dropdown,
        "quality": quality_dropdown,
        "unit_range": units_slider
    }
)

display(out)


Output()